# Unidad 4 · Práctica — Gramática de gráficos con Altair

Ciencia de Datos · UTN FRM · 2026

La única clase práctica de la unidad. Se corre **de arriba hacia abajo**: cada sección usa lo que
dejaron las anteriores. Cada título es una celda propia, así que el índice de Colab sirve para
saltar entre secciones y plegarlas.

Altair está construido sobre la misma idea que la teoría: un gráfico no se dibuja, **se
declara**. No se le dice a la computadora cómo trazar cada punto; se le dice qué datos usar, con
qué marca, qué variable va a cada atributo visual, qué transformar y cómo componer. Cada parte de
este notebook es uno de esos componentes, en el mismo orden en que aparecen en la teoría, y al
final se juntan todos en un póster que responde una pregunta con datos del canónico.

## Hoja de ruta

| Parte | Componente de la gramática | Qué se escribe en Altair |
|---|---|---|
| **1** | Datos | `alt.Chart(tabla)` |
| **2** | Marcas y mapeos estéticos | `mark_*()` y `encode(...)`, con el tipo de cada variable |
| **3** | Transformaciones estadísticas | agregaciones en la codificación y `transform_*()` |
| **4** | Escalas y sistema de coordenadas | `alt.Scale(...)` dentro de cada canal |
| **5** | Composición | capas (`+`), facetas, repeticiones, concatenaciones (`\|`, `&`) |
| **6** | Interactividad | parámetros y selecciones |
| **7** | Todo junto: un póster | los seis componentes al servicio de una pregunta |

Las secciones se numeran por parte: «la sección 3.2» es la segunda de la Parte 3.

## Antes de empezar

Hace falta el mismo `fifa_canonico_2026-07-23.csv` de siempre; la primera celda lo baja si no
está. La segunda instala lo necesario para guardar gráficos como PNG (sección 7.7).

In [1]:
# El dataset se baja del Drive de la cátedra la primera vez que se corre el notebook
import os

ARCHIVO  = "fifa_canonico_2026-07-23.csv"
ID_DRIVE = "1cF30_fSvxNKpK5-AKJlzUqLHZTTUf0ls"

# el ! corre un comando de la terminal desde la celda, y $ le pasa una variable de Python;
# -O es el nombre con el que gdown guarda el archivo
if not os.path.exists(ARCHIVO):
    !gdown $ID_DRIVE -O $ARCHIVO

Downloading...
From: https://drive.google.com/uc?id=1cF30_fSvxNKpK5-AKJlzUqLHZTTUf0ls
To: /content/fifa_canonico_2026-07-23.csv
100% 9.56M/9.56M [00:01<00:00, 7.04MB/s]


In [2]:
# para guardar gráficos como PNG (-q: instala sin llenar la salida de mensajes)
!pip install -q vl-convert-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 33.4 MB/s eta 0:00:00


In [3]:
import pandas as pd
import numpy as np
import altair as alt

# cuántas columnas y qué ancho muestra pandas al imprimir una tabla
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

# el canónico tiene casi 19.000 filas y Altair corta en 5.000; la sección 1.2 explica por qué
alt.data_transformers.disable_max_rows()

SEMILLA = 42   # la misma muestra en cada corrida
# low_memory=False: pandas lee cada columna entera antes de decidir su tipo, en vez de adivinarlo por partes
df = pd.read_csv(ARCHIVO, low_memory=False)
df.shape

(18936, 90)

## Parte 1 — Datos

Todo gráfico empieza por una tabla: una fila por observación, una columna por variable. En
Altair, la tabla es lo primero que se escribe —`alt.Chart(tabla)`— y todo lo demás se declara
**sobre** ella: qué columna va a cada eje, qué se agrupa, qué se filtra. Por eso conviene
llegar al gráfico con la tabla bien armada: con las columnas que importan y nombres que se lean
solos.

### 1.1 Una fila por jugador

In [4]:
# las columnas de hoy, con nombres que se leen solos
# ROL traduce la posición del juego (best_position) a siete roles en castellano
ROL = {"GK": "Arquero", "CB": "Defensor central",
       "LB": "Lateral", "RB": "Lateral", "LWB": "Lateral", "RWB": "Lateral",
       "CDM": "Volante defensivo",
       "CM": "Volante", "CAM": "Volante", "LM": "Volante", "RM": "Volante",
       "LW": "Extremo", "RW": "Extremo", "ST": "Delantero", "CF": "Delantero"}

jug = pd.DataFrame({
    "nombre":    df["short_name"],
    "rol":       df["best_position"].map(ROL),   # .map reemplaza cada código por su valor en el diccionario
    "liga":      df["league_name"] + " (" + df["league_country"] + ")",
    "edad":      df["age"],
    "overall":   df["overall"],
    "valor":     df["value_eur"] / 1e6,        # millones de euros
    "estrellas": df["skill_moves"],            # gambeta, de 1 a 5
    "velocidad": df["movement_sprint_speed"],
    "fuerza":    df["power_strength"],
    "regate":    df["skill_dribbling"],
})
jug.head()

,nombre,rol,liga,edad,overall,valor,estrellas,velocidad,fuerza,regate
0,P. Billing,Volante,Superliga (Denmark),29,76,7.5,3,66,79,74
1,A. Şimşir,Volante,Superliga (Denmark),23,76,12.0,5,77,69,78
2,G. Links,Volante,Superliga (Denmark),26,75,6.0,4,89,62,72
3,M. Elyounoussi,Volante,Superliga (Denmark),30,75,5.5,4,75,68,77
4,D. Kotarski,Arquero,Superliga (Denmark),25,75,6.0,1,44,62,33


Once jugadores tienen valor de mercado cero. Para preguntas sobre el valor son ruido, y en la
escala logarítmica de la Parte 4 no tienen lugar: se sacan acá, una sola vez. Y para los
gráficos de puntos, donde 19.000 marcas se encimarían, se arma una muestra fija de 2.000.

In [5]:
# sin valor de mercado (sumar una comparación cuenta los True)
(jug["valor"] == 0).sum()

np.int64(11)

In [6]:
# se sacan, y se arma la muestra para los gráficos de puntos
# reset_index(drop=True) renumera las filas desde 0 sin guardar la numeración vieja como columna
jug = jug[jug["valor"] > 0].reset_index(drop=True)
# random_state fija el azar: con la misma semilla sale siempre la misma muestra
muestra = jug.sample(2000, random_state=SEMILLA)
pd.Series({"jugadores": len(jug), "muestra": len(muestra)})

,0
jugadores,18925
muestra,2000


### 1.2 El gráfico lleva sus datos adentro

Lo que Altair produce no es una imagen: es una **especificación**, un JSON que describe el
gráfico, y la tabla viaja adentro. Se ve con un gráfico mínimo —las marcas y la codificación son
el tema de la Parte 2— pasándole cuatro tablas distintas para dibujar lo mismo:

In [7]:
# el mismo gráfico de barras, con cuatro formas de pasarle los datos
# (median(valor) es una agregación dentro del canal: la sección 3.1 la explica)
def barras(tabla):
    return alt.Chart(tabla).mark_bar().encode(x="median(valor):Q", y="rol:N")

# assign agrega columnas a una copia de la tabla; as_index=False deja el rol como columna y no como índice
opciones = {"df entero (90 columnas)": barras(df.assign(rol=df["best_position"].map(ROL), valor=df["value_eur"] / 1e6)),
            "jug (10 columnas)":       barras(jug),
            "sólo rol y valor":        barras(jug[["rol", "valor"]]),
            "ya agregado":             barras(jug.groupby("rol", as_index=False)["valor"].median())}

In [8]:
# tamaño de cada especificación, en MB
# to_json() devuelve la especificación como texto; su largo en caracteres es, a grandes rasgos, su peso en bytes
pd.Series({k: len(g.to_json()) / 1e6 for k, g in opciones.items()}).round(3)

,0
df entero (90 columnas),58.696
jug (10 columnas),5.291
sólo rol y valor,1.276
ya agregado,0.001


Siete barras, de 58 MB a 1 KB. Por eso Altair corta en 5.000 filas si no se le dice lo contrario,
y por eso de acá en adelante cada gráfico recibe **sólo las columnas que usa**. Fijate además que
la última versión funcionó igual con la tabla ya agregada: `median(valor)` de una sola fila por
rol es esa fila. La agregación puede vivir en pandas o en el gráfico; la Parte 3 vuelve sobre
esto.

## Parte 2 — Marcas y mapeos estéticos

Con la tabla elegida, quedan dos decisiones: **qué forma** representa a cada fila —la marca, o
geometría: puntos, barras, líneas— y **qué variable va a cada atributo visual** —el mapeo
estético: posición horizontal, vertical, color, tamaño—. En Altair son dos llamadas, `mark_*()`
y `encode(...)`, y juntas con los datos ya forman una oración completa.

### 2.1 La oración mínima

In [9]:
# datos, marca y mapeo: overall a la posición horizontal, valor a la vertical
alt.Chart(muestra).mark_point().encode(x="overall", y="valor")

alt.Chart(...)

No se escribió nada sobre ejes, rangos, colores ni tamaños: eso lo resuelve Vega-Lite, la
biblioteca sobre la que está construido Altair. Lo que sí se escribió es exactamente lo que
importa, y se puede leer en la especificación:

In [10]:
# la especificación, sin la tabla
# to_dict() devuelve la especificación como diccionario de Python; se muestran sólo dos claves
spec = alt.Chart(muestra).mark_point().encode(x="overall", y="valor").to_dict()
{k: v for k, v in spec.items() if k in ("mark", "encoding")}

{'mark': {'type': 'point'},
 'encoding': {'x': {'field': 'overall', 'type': 'quantitative'},
  'y': {'field': 'valor', 'type': 'quantitative'}}}

Una marca y dos canales. Y algo que no se escribió: `"type": "quantitative"`. Altair adivinó el
tipo de cada variable mirando la columna; la sección 2.3 muestra por qué conviene no dejarlo
adivinar.

### 2.2 La marca es una palabra

Como el gráfico es una declaración, cambiar la geometría es cambiar una palabra: el mapeo queda
igual. Abajo, el overall de cada rol con tres marcas distintas. La tercera, `mark_boxplot`, es
una marca que trae su propia transformación: calcula cuartiles antes de dibujar.

In [11]:
# el mismo mapeo, tres marcas
# properties fija el tamaño de cada gráfico en píxeles; title, su título
base = alt.Chart(muestra[["rol", "overall"]]).encode(x="overall:Q", y="rol:N").properties(width=230, height=180)

# alt.hconcat pone gráficos lado a lado (es composición: sección 5.4)
# un argumento de la marca, como opacity=0.3, es un valor fijo para todos los puntos;
# en encode, en cambio, el atributo sale de una columna
alt.hconcat(base.mark_tick().properties(title="mark_tick"),
            base.mark_circle(opacity=0.3).properties(title="mark_circle"),
            base.mark_boxplot().properties(title="mark_boxplot"))

alt.HConcatChart(...)

Las tres responden la misma pregunta con distinto detalle: las dos primeras muestran cada
jugador, la caja resume. Ninguna es la correcta en general; la marca se elige por lo que el
lector tiene que ver.

### 2.3 El tipo de cada variable

Cada variable del mapeo se declara con un tipo: `:Q` cuantitativa, `:O` ordinal, `:N` nominal,
`:T` temporal. El tipo decide la escala, el eje y la leyenda. `estrellas` —la gambeta, de 1 a 5—
admite los tres primeros; abajo, la misma columna al color, tres veces.

In [12]:
# la misma columna al color, con tres tipos
# size=20 en la marca: el mismo tamaño fijo para todos los puntos
base = alt.Chart(muestra[["overall", "valor", "estrellas"]]).mark_circle(size=20).encode(
    x="overall:Q", y="valor:Q").properties(width=230, height=180)

alt.hconcat(base.encode(color="estrellas:Q").properties(title=":Q cuantitativa"),
            base.encode(color="estrellas:O").properties(title=":O ordinal"),
            base.encode(color="estrellas:N").properties(title=":N nominal"))

alt.HConcatChart(...)

`:Q` arma un degradé continuo, con una leyenda que sugiere valores que no existen. `:N` da cinco
colores sin orden, como si 2 y 5 estrellas no tuvieran relación. `:O` da cinco pasos ordenados,
que es lo que la variable es. Desde acá, cada variable se escribe con su tipo.

### 2.4 Más canales

`x` e `y` son canales; también `color`, `size`, `shape` y `opacity`. Cada uno agrega una variable,
pero no todos se leen igual: la posición se compara con mucha precisión, y el color o el tamaño
con mucha menos. Lo más importante de la pregunta va a la posición.

In [13]:
# dos variables más: el rol al color y la edad al tamaño
alt.Chart(muestra).mark_circle(opacity=0.5).encode(
    x="overall:Q", y="valor:Q", color="rol:N", size="edad:Q")

alt.Chart(...)

Entraron cuatro variables y las dos nuevas no se leen: siete colores encimados y tamaños que
apenas se distinguen. Un canal más no es información más; la sección 5.2 resuelve lo mismo con
un panel por rol. Y en todos estos gráficos hay otro problema, que es de la escala y no del
mapeo: casi todos los puntos están aplastados contra el piso. Es la Parte 4.

### 2.5 Forma corta y forma extendida

`x="overall:Q"` es la forma corta: «esta columna, a este canal, con todo lo demás por defecto».
Cuando hace falta decir algo más del canal —el título del eje, el orden, la escala— se usa la
forma extendida, `alt.X(...)`, `alt.Y(...)`, `alt.Color(...)`. Es el mismo mapeo con más
detalle, y las dos formas conviven en un mismo gráfico.

In [14]:
# forma extendida: título del eje y orden de las barras
# count() cuenta filas: no necesita columna
# sort="-x" ordena las barras por el valor del eje x, de mayor a menor (sin el menos, de menor a mayor)
# title=None saca el título del eje
alt.Chart(jug[["rol"]]).mark_bar().encode(
    x=alt.X("count():Q", title="Jugadores"),
    y=alt.Y("rol:N", sort="-x", title=None))

alt.Chart(...)

## Parte 3 — Transformaciones estadísticas

Muchas veces no se grafica la tabla tal como está, sino un resumen: un promedio por grupo, un
conteo por intervalo, una curva que suaviza la nube. Esas son las **transformaciones
estadísticas**, y en la gramática ocupan el lugar entre los datos y la marca. En Altair se
declaran dentro del gráfico: no modifican la tabla, le dicen a Vega-Lite qué calcular antes de
dibujar.

### 3.1 Agregar dentro del mapeo

In [15]:
# valor medio por rol: la agregación se escribe en el canal
alt.Chart(jug[["rol", "valor"]]).mark_bar().encode(
    x="mean(valor):Q", y=alt.Y("rol:N", sort="-x"))

alt.Chart(...)

`mean(valor)` agrupa por lo que esté en los otros canales —acá, el rol— y promedia. Pero el
promedio es una elección entre varias, y con el valor, que tiene unos pocos jugadores
carísimos, pesa. Antes de quedarse con el gráfico, medir:

In [16]:
# media y mediana del valor por rol
jug.groupby("rol")["valor"].agg(["mean", "median"]).round(2)

,mean,median
rol,,
Arquero,1.84,0.50
Defensor central,2.68,1.00
Delantero,2.92,1.00
Extremo,3.85,1.20
Lateral,2.50,0.98
Volante,3.57,1.20
Volante defensivo,3.32,1.10


La media es entre dos y casi cuatro veces la mediana en todos los roles: los cracks la arrastran.
Un «valor típico» es la mediana, y en la declaración cambia una palabra:

In [17]:
# lo mismo, con la mediana
alt.Chart(jug[["rol", "valor"]]).mark_bar().encode(
    x="median(valor):Q", y=alt.Y("rol:N", sort="-x"))

alt.Chart(...)

### 3.2 Agregar como un paso propio

La agregación también se puede escribir como un paso separado, con `transform_aggregate`, que
crea columnas nuevas con nombre propio. Sirve cuando el resumen se usa en más de un canal o en
más de una marca. Como esas columnas no existen en la tabla, Altair no puede adivinar su tipo:
hay que declararlo.

In [18]:
# por rol: mediana del valor, overall medio y cantidad de jugadores, en un paso
# cada argumento con nombre crea una columna: nombre_nuevo="agregación(columna)";
# groupby dice por qué columnas se agrupa
alt.Chart(jug[["rol", "valor", "overall"]]).transform_aggregate(
    valor_tipico="median(valor)", overall_medio="mean(overall)", jugadores="count()",
    groupby=["rol"]
).mark_circle().encode(
    x=alt.X("overall_medio:Q", scale=alt.Scale(zero=False)),
    y=alt.Y("valor_tipico:Q", scale=alt.Scale(zero=False)),
    size="jugadores:Q", color="rol:N")

alt.Chart(...)

Cada punto es un rol: su overall medio, su valor típico y, en el tamaño, cuántos jugadores tiene.
Es la misma tabla que daría un `groupby` de pandas, declarada adentro del gráfico. Los ejes no
arrancan en cero (`zero=False`): con siete puntos en un rango chico, quedarían amontonados en una
esquina. Por qué eso se puede hacer con puntos y no con barras es la sección 4.2.

### 3.3 Filtrar y calcular

`transform_filter` saca filas y `transform_calculate` crea columnas, las dos con expresiones
sobre `datum`, que es la fila. Abajo, sin arqueros, con la edad partida en tres tramos y la
mediana del valor de cada tramo para cada overall. El eje del valor va en escala logarítmica,
que se explica en la sección 4.1: por ahora alcanza con saber que separa las tres líneas.

In [19]:
# filtro, columna calculada y agregación: tres transformaciones encadenadas
# alt.datum.rol != "Arquero" se escribe en Python, pero Altair lo traduce a una expresión de Vega-Lite
# en transform_calculate la expresión va como texto: condición ? valor_si_sí : valor_si_no, encadenadas
alt.Chart(jug[["rol", "edad", "overall", "valor"]]).transform_filter(
    alt.datum.rol != "Arquero"
).transform_calculate(
    tramo="datum.edad < 23 ? '1 · hasta 22' : datum.edad < 30 ? '2 · de 23 a 29' : '3 · 30 o más'"
).mark_line().encode(
    x=alt.X("overall:Q", scale=alt.Scale(zero=False)),
    y=alt.Y("median(valor):Q", scale=alt.Scale(type="log")),
    color="tramo:N")

alt.Chart(...)

A igual overall, la línea de los más jóvenes va arriba y la de los de 30 o más abajo: el mercado
paga lo que el jugador todavía puede llegar a ser. El hallazgo salió de una columna que no
estaba en la tabla: la declaró el gráfico.

### 3.4 Intervalos

`bin` agrupa una variable cuantitativa en intervalos. Con `count()` es un histograma; con otra
agregación, una curva resumida.

In [20]:
# histograma del overall, en intervalos de 2
# step fija el ancho de cada intervalo; con maxbins=20 se fijaría, en cambio, cuántos hay como máximo
alt.Chart(jug[["overall"]]).mark_bar().encode(
    x=alt.X("overall:Q", bin=alt.Bin(step=2)), y="count():Q")

alt.Chart(...)

In [21]:
# valor mediano por intervalo de edad
# point=True agrega un punto en cada vértice de la línea
alt.Chart(jug[["edad", "valor"]]).mark_line(point=True).encode(
    x=alt.X("edad:Q", bin=alt.Bin(step=2)), y="median(valor):Q")

alt.Chart(...)

La curva confirma lo de la sección 3.3 desde otro lado: el valor mediano sube hasta los 22 o 23
años y después cae.

### 3.5 Suavizar

`transform_loess` ajusta una curva suave a una nube de puntos: pequeñas regresiones locales a lo
largo del eje x, sin imponer una forma de antemano. `transform_regression` hace lo mismo con una
forma fija, una recta por defecto. Abajo, el overall según la edad.

In [22]:
# la tendencia del overall con la edad: LOESS y una recta
nube_edad = alt.Chart(muestra[["edad", "overall"]])

# los dos argumentos son la columna x y la columna y; la curva sale con esos mismos nombres de columna
alt.hconcat(nube_edad.transform_loess("edad", "overall").mark_line().encode(x="edad:Q", y=alt.Y("overall:Q", scale=alt.Scale(zero=False))).properties(title="LOESS"),
            nube_edad.transform_regression("edad", "overall").mark_line().encode(x="edad:Q", y=alt.Y("overall:Q", scale=alt.Scale(zero=False))).properties(title="regresión lineal"))

alt.HConcatChart(...)

La recta dice «con la edad, el overall sube» y no dice más. La curva cuenta mejor la historia:
sube hasta fines de los veinte y se aplana. La transformación también es una decisión, y la
forma que se impone de antemano puede esconder lo que hay. En la sección 5.1 la curva va arriba
de sus puntos.

## Parte 4 — Escalas y sistema de coordenadas

Cada canal tiene una **escala**: la función que traduce un valor de los datos a una posición, un
color o un tamaño. Altair elige una por defecto según el tipo de la variable, pero dos decisiones
son siempre del autor: si la escala es lineal o logarítmica, y dónde empieza el eje. Las dos se
declaran dentro del canal, con `alt.Scale(...)`.

### 4.1 Lineal o logarítmica

In [23]:
# el mismo gráfico, con el valor en escala lineal y en logarítmica
puntos = alt.Chart(muestra[["overall", "valor"]]).mark_circle(size=15, opacity=0.5).properties(width=300, height=230)

alt.hconcat(puntos.encode(x="overall:Q", y="valor:Q").properties(title="lineal"),
            puntos.encode(x="overall:Q", y=alt.Y("valor:Q", scale=alt.Scale(type="log"))).properties(title="logarítmica"))

alt.HConcatChart(...)

In [24]:
# qué parte de los jugadores vale menos de 5 millones (el promedio de True y False es una proporción)
(jug["valor"] < 5).mean().round(3)

np.float64(0.893)

Nueve de cada diez jugadores valen menos de 5 millones, y en la escala lineal quedan aplastados
contra el piso: el gráfico sólo muestra a los cracks. En la logarítmica cada escalón es
multiplicar por diez, y aparece la forma de la relación: casi una recta, que quiere decir que el
valor se **multiplica** con cada punto de overall.

### 4.2 Dónde empieza el eje

Una barra codifica un número con su **largo**, así que tiene que empezar en cero: si no, el largo
miente. Un punto codifica con su **posición**, y su eje puede empezar donde convenga. La regla no
es sobre la escala sola: es sobre la escala y la marca juntas.

In [25]:
# overall medio por rol: barras con el eje cortado, barras desde cero, y puntos
ov = jug.groupby("rol", as_index=False)["overall"].mean()
b = alt.Chart(ov).encode(y=alt.Y("rol:N", sort="-x")).properties(width=200, height=180)

# domain=[63, 68] fija dónde empieza y termina el eje; clip=True recorta lo que se sale de ese rango
# filled=True rellena los puntos, que mark_point dibuja huecos por defecto
alt.hconcat(b.mark_bar(clip=True).encode(x=alt.X("overall:Q", scale=alt.Scale(domain=[63, 68]))).properties(title="barras, eje desde 63"),
            b.mark_bar().encode(x="overall:Q").properties(title="barras desde cero"),
            b.mark_point(filled=True, size=80).encode(x=alt.X("overall:Q", scale=alt.Scale(zero=False))).properties(title="puntos, eje cortado"))

alt.HConcatChart(...)

In [26]:
# las diferencias reales
ov.set_index("rol")["overall"].round(1).sort_values()

,overall
rol,
Arquero,64.1
Delantero,65.2
Lateral,65.7
Volante,65.8
Defensor central,65.9
Volante defensivo,66.9
Extremo,67.6


Entre el rol con menos overall y el con más hay 3,5 puntos. Las barras cortadas lo muestran como
si los arqueros fueran la mitad de buenos; las barras desde cero esconden la diferencia; los
puntos con el eje cortado la muestran del tamaño que tiene. Por la misma razón, en los gráficos
de puntos de la Parte 2 la mitad izquierda estaba vacía: el eje del overall arrancaba en 0.

In [27]:
# los dos canales que se usan de acá en adelante
X_OV  = alt.X("overall:Q", scale=alt.Scale(zero=False))
Y_LOG = alt.Y("valor:Q", scale=alt.Scale(type="log"))

alt.Chart(muestra[["overall", "valor"]]).mark_circle(size=15, opacity=0.5).encode(x=X_OV, y=Y_LOG)

alt.Chart(...)

`X_OV` e `Y_LOG` son canales guardados en variables: como el gráfico es una declaración, sus
partes se pueden nombrar y reusar. Es lo que permite, en la Parte 7, que todos los paneles de un
póster compartan las mismas decisiones.

## Parte 5 — Composición

Un gráfico solo responde poco. La gramática compone: **capas** que se superponen en el mismo
sistema de coordenadas, **facetas** que repiten el gráfico por grupo, **repeticiones** que
cambian la variable, y **concatenaciones** que ponen gráficos distintos lado a lado. En Altair
cada composición es un operador sobre gráficos ya declarados.

### 5.1 Capas

In [28]:
# los puntos y la curva LOESS de la sección 3.5, en el mismo espacio
# color y strokeWidth (grosor de la línea) van en la marca: son fijos, no salen de ninguna columna
puntos_edad = nube_edad.mark_circle(size=12, opacity=0.25, color="gray").encode(
    x="edad:Q", y=alt.Y("overall:Q", scale=alt.Scale(zero=False)))
curva_edad  = nube_edad.transform_loess("edad", "overall").mark_line(color="#009086", strokeWidth=3).encode(
    x="edad:Q", y="overall:Q")

puntos_edad + curva_edad

alt.LayerChart(...)

El `+` es `alt.layer(...)`. Cada capa tiene su marca y sus transformaciones —la nube, ninguna; la
curva, un LOESS— y comparten los ejes. La curva dice lo típico; la nube, cuánto se apartan los
jugadores de lo típico. Con la curva sola, la sección 3.5 no dejaba ver eso.

### 5.2 Facetas

In [29]:
# el mismo gráfico, un panel por rol
# legend=None saca la leyenda: el título de cada panel ya dice el rol
# en una faceta, width y height son el tamaño de cada panel; columns=4 corta en filas de cuatro
alt.Chart(muestra[["overall", "valor", "rol"]]).mark_circle(size=12, opacity=0.5).encode(
    x=X_OV, y=Y_LOG, color=alt.Color("rol:N", legend=None)
).properties(width=150, height=120).facet("rol:N", columns=4)

alt.FacetChart(...)

Es la salida para el gráfico ilegible de la sección 2.4: en vez de siete colores encimados, siete
paneles con los mismos ejes. La faceta también puede cruzar dos variables, una en las filas y
otra en las columnas; abajo, los tramos de edad de la sección 3.3 contra cuatro roles.

In [30]:
# filas: tramo de edad; columnas: rol
cuatro = ["Defensor central", "Volante", "Extremo", "Delantero"]
# .isin se queda con las filas cuyo rol está en la lista
alt.Chart(muestra[muestra["rol"].isin(cuatro)][["overall", "valor", "rol", "edad"]]).transform_calculate(
    tramo="datum.edad < 23 ? '1 · hasta 22' : datum.edad < 30 ? '2 · de 23 a 29' : '3 · 30 o más'"
).mark_circle(size=10, opacity=0.5).encode(
    x=X_OV, y=Y_LOG
).properties(width=130, height=90).facet(row="tramo:N", column="rol:N")

alt.FacetChart(...)

### 5.3 Repetir

In [31]:
# tres atributos contra el valor, sin arqueros
# alt.repeat("column") ocupa el lugar del nombre de la columna; como cambia en cada panel,
# Altair no puede adivinar el tipo y hay que darlo con type=
alt.Chart(muestra[muestra["rol"] != "Arquero"][["velocidad", "fuerza", "regate", "valor"]]).mark_circle(size=10, opacity=0.3).encode(
    x=alt.X(alt.repeat("column"), type="quantitative"), y=Y_LOG
).properties(width=200, height=180).repeat(column=["velocidad", "fuerza", "regate"])

alt.RepeatChart(...)

`repeat` cambia la variable de un canal y deja todo lo demás igual: es la misma oración con una
palabra distinta en cada panel. Sirve para comparar atributos de un vistazo; lo que se ve a ojo,
conviene medirlo:

In [32]:
# correlación de cada atributo con el logaritmo del valor, sin arqueros, con todos los jugadores
# corrwith calcula la correlación de cada columna con una misma serie; log10, porque el valor se lee en escala log
campo = jug[jug["rol"] != "Arquero"]
campo[["velocidad", "fuerza", "regate"]].corrwith(np.log10(campo["valor"])).round(2)

,0
velocidad,0.34
fuerza,0.26
regate,0.61


El regate acompaña al valor bastante más que la velocidad o la fuerza: 0,61 contra 0,34 y 0,26.
El gráfico mostró la forma; el número dice cuánto.

### 5.4 Concatenar, y compartir o no la escala

`hconcat` y `vconcat` —o `|` y `&`— ponen gráficos distintos lado a lado. Cuando los dos miden lo
mismo, hay una decisión que suele pasar desapercibida: si comparten la escala o cada uno tiene la
suya. Abajo, el valor mediano por rol en dos ligas.

In [33]:
# valor mediano por rol en dos ligas, cada panel con su escala
# title= en alt.Chart es otra forma de ponerle título, sin pasar por properties
def por_rol(liga):
    datos = jug[jug["liga"] == liga].groupby("rol", as_index=False)["valor"].median()
    return alt.Chart(datos, title=liga).mark_bar().encode(
        x="valor:Q", y=alt.Y("rol:N", sort="-x")).properties(width=220, height=170)

# | es alt.hconcat, y & sería alt.vconcat
dos_ligas = por_rol("Premier League (England)") | por_rol("Liga Profesional de Fútbol (Argentina)")
dos_ligas

alt.HConcatChart(...)

In [34]:
# lo mismo, con la escala compartida
# resolve_scale decide, canal por canal, si los paneles comparten escala ("shared") o no ("independent")
dos_ligas.resolve_scale(x="shared")

alt.HConcatChart(...)

Con una escala por panel, las barras de las dos ligas tienen largos parecidos y las ligas parecen
comparables. Con la escala compartida aparece la diferencia real. Cuando dos paneles se van a
comparar, la escala se comparte; cuando no, se dice en el título.

## Parte 6 — Interactividad

La interactividad no es un agregado aparte: es un componente más de la misma gramática. Se
declara un **parámetro** —un valor que cambia con el mouse o con un control— y después se lo usa
donde se usaría cualquier otra cosa: en un filtro, en una condición de color, en la opacidad.
No se escribe qué pasa con cada clic; se declara de qué depende el gráfico.

### 6.1 Tooltips y zoom

In [35]:
# pasar el mouse muestra el jugador; la rueda y el arrastre hacen zoom
# tooltip recibe la lista de columnas que se muestran, en ese orden
alt.Chart(muestra[["nombre", "rol", "liga", "overall", "valor"]]).mark_circle(size=25, opacity=0.6).encode(
    x=X_OV, y=Y_LOG, color="rol:N", tooltip=["nombre", "rol", "liga", "overall", "valor"]
).interactive()

alt.Chart(...)

El tooltip es un canal más del mapeo, como `x` o `color`: no hace falta ningún parámetro.
`.interactive()` sí agrega uno, que ata las escalas de los ejes al zoom y al arrastre.

### 6.2 Una selección que resalta

In [36]:
# cliquear un rol en la leyenda lo resalta y apaga el resto
# fields=["rol"]: lo que se guarda al cliquear es el rol, y quedan seleccionados todos los puntos de ese rol
# bind="legend": el clic se hace sobre la leyenda y no sobre los puntos
rol_elegido = alt.selection_point(fields=["rol"], bind="legend")

# alt.value(...) es un valor fijo, no una columna; add_params agrega el parámetro al gráfico, sin eso no existe
alt.Chart(muestra[["overall", "valor", "rol"]]).mark_circle(size=25).encode(
    x=X_OV, y=Y_LOG, color="rol:N",
    opacity=alt.condition(rol_elegido, alt.value(0.8), alt.value(0.05))
).add_params(rol_elegido)

alt.Chart(...)

`selection_point` guarda lo que se cliqueó; `alt.condition` lo lee: si el punto está en la
selección, opacidad 0,8; si no, 0,05. Es un mapeo estético que depende de un parámetro.

### 6.3 Un menú

In [37]:
# un menú con las cinco ligas con más jugadores filtra los puntos
# value_counts ordena de mayor a menor, y .index[:5] se queda con los nombres de las cinco primeras
ligas = jug["liga"].value_counts().index[:5].tolist()
# value= es lo que muestra el gráfico antes de tocar el menú; name= es el rótulo al lado del control
liga_elegida = alt.param(value=ligas[0], bind=alt.binding_select(options=ligas, name="Liga "))

alt.Chart(jug[jug["liga"].isin(ligas)][["liga", "nombre", "overall", "valor", "rol"]]).mark_circle(size=30, opacity=0.6).encode(
    x=X_OV, y=Y_LOG, color="rol:N", tooltip=["nombre", "overall", "valor"]
).add_params(liga_elegida).transform_filter(alt.datum.liga == liga_elegida)

alt.Chart(...)

El parámetro está atado a un menú, y el filtro de la sección 3.3 lo usa en lugar de un valor
fijo. Es la misma transformación; lo que cambió es de dónde sale el valor.

### 6.4 Un slider

In [38]:
# un slider elige la edad; la curva de todos queda de fondo
# min, max y step: el rango del slider y de a cuánto salta
edad_elegida = alt.param(value=20, bind=alt.binding_range(min=17, max=38, step=1, name="Edad "))

# el fondo es la mediana del valor para cada overall, calculada con pandas, en gris
fondo = alt.Chart(jug.groupby("overall", as_index=False)["valor"].median()).mark_line(color="lightgray", strokeWidth=3).encode(x=X_OV, y=Y_LOG)
de_esa_edad = alt.Chart(jug[["edad", "overall", "valor", "nombre"]]).mark_circle(size=20, opacity=0.6, color="#009086").encode(
    x=X_OV, y=Y_LOG, tooltip=["nombre", "edad", "overall", "valor"]
).add_params(edad_elegida).transform_filter(alt.datum.edad == edad_elegida)

fondo + de_esa_edad

alt.LayerChart(...)

Moviendo el slider, los jugadores jóvenes quedan por encima de la curva gris y los veteranos por
debajo: el hallazgo de la sección 3.3, ahora para explorar edad por edad.

### 6.5 Vistas vinculadas

Una selección de un gráfico puede filtrar otro. Arrastrando sobre la nube se marca un rectángulo
—un *brush*—, y las barras de al lado cuentan sólo a los jugadores que quedaron adentro.

In [39]:
# arrastrar sobre la nube; las barras cuentan lo seleccionado
brush = alt.selection_interval()

# alt.condition(brush, "rol:N", alt.value("lightgray")): adentro del rectángulo el color sale del rol;
# afuera, gris fijo
nube_sel = alt.Chart(muestra[["overall", "valor", "rol"]]).mark_circle(size=20).encode(
    x=X_OV, y=Y_LOG, color=alt.condition(brush, "rol:N", alt.value("lightgray"))
).add_params(brush).properties(width=360, height=260)

# el parámetro se declara en un gráfico (add_params) y se lee en otro (transform_filter)
conteo = alt.Chart(muestra[["overall", "valor", "rol"]]).mark_bar().encode(
    x="count():Q", y=alt.Y("rol:N", sort="-x"), color="rol:N"
).transform_filter(brush).properties(width=200, height=260)

nube_sel | conteo

alt.HConcatChart(...)

El mismo parámetro sirve para dos cosas en dos gráficos: en la nube, una condición de color; en
las barras, un filtro. Con esto un gráfico deja de contar una historia y pasa a ser una
herramienta para buscarla.

## Parte 7 — Todo junto: un póster

Hasta acá, componentes sueltos. Un póster los usa todos para **responder una pregunta**, y la
respuesta interesante casi nunca es la obvia. La pregunta: **¿la gambeta se paga?** Cada sección
que sigue es una decisión sobre uno de los componentes, y dice cuál.

### 7.1 La primera oración

Datos, una agregación, una marca y dos canales: la versión que se le ocurre a cualquiera.

In [40]:
# valor mediano según las estrellas de gambeta
alt.Chart(jug[["estrellas", "valor"]]).mark_bar().encode(x="estrellas:O", y="median(valor):Q")

alt.Chart(...)

In [41]:
# los números de esas barras
jug.groupby("estrellas")["valor"].median()

,valor
estrellas,
1,0.500
2,0.775
3,1.400
4,3.100
5,12.500


De 2 a 5 estrellas, el valor mediano pasa de 0,78 a 12,5 millones: dieciséis veces. La lectura
obvia es que la gambeta se paga carísimo. Antes de creerla, mirar los **datos**: quiénes son los
de cada grupo.

### 7.2 Mirar los datos

In [42]:
# cuántos hay en cada grupo y qué overall tienen (size cuenta las filas del grupo)
jug.groupby("estrellas")["overall"].agg(["size", "median"])

,size,median
estrellas,,
1,2106,64.0
2,9135,64.0
3,6533,67.0
4,1098,72.0
5,53,77.0


In [43]:
# quiénes tienen 1 estrella
# crosstab cuenta cuántas filas hay en cada combinación; rename les pone nombre a las columnas True y False
pd.crosstab(jug["estrellas"], jug["rol"] == "Arquero").rename(columns={False: "de campo", True: "arqueros"})

rol,de campo,arqueros
estrellas,,
1,0,2106
2,9135,0
3,6533,0
4,1098,0
5,53,0


Dos problemas. Los de más estrellas son también **mucho mejores**: overall mediano de 77 contra
64 de los de 2 estrellas, y el valor crece con el overall (sección 4.1). Y la barra de 1 estrella
no son jugadores torpes: **son todos los arqueros y nadie más**. La comparación honesta es otra:
sin arqueros, y a igual overall.

### 7.3 Otra transformación, otra escala

La oración cambia en dos componentes. La **transformación**: filtrar los arqueros y agregar por
overall y estrellas, en vez de sólo por estrellas. La **escala**: logarítmica para el valor. Se
quedan los grupos de 20 jugadores o más, para que cada punto de la curva sea una mediana de algo.

In [44]:
# valor mediano por overall y estrellas, sin arqueros, sólo grupos de 20 jugadores o más
# agg(valor="median", n="size"): dos columnas nuevas con nombre propio, la mediana y el tamaño del grupo
curvas = (jug[jug["rol"] != "Arquero"]
          .groupby(["overall", "estrellas"])["valor"].agg(valor="median", n="size")
          .reset_index())
curvas = curvas[curvas["n"] >= 20]
curvas.groupby("estrellas")["overall"].agg(["min", "max", "size"])

,min,max,size
estrellas,,,
2,47,81,35
3,55,83,29
4,63,83,20


In [45]:
# la oración nueva
alt.Chart(curvas).mark_line(point=True).encode(x=X_OV, y=Y_LOG, color="estrellas:O")

alt.Chart(...)

In [46]:
# cuánto valen a overall 80, según las estrellas
curvas[curvas["overall"] == 80].set_index("estrellas")["valor"]

,valor
estrellas,
2,21.0
3,20.5
4,23.5


Los de 5 estrellas no llegan a 20 en ningún overall: quedan 2, 3 y 4. Y las curvas se pisan: a
overall 80, un jugador vale entre 20,5 y 23,5 millones con cualquiera de las tres. **La gambeta
no se paga; se paga el nivel.** Esa es la respuesta; falta contarla.

### 7.4 Un mapeo para todo el póster

El póster va a tener tres paneles, y en los tres las estrellas van al color. Si cada panel tiene
su escala de color, el lector reaprende la leyenda en cada uno. Se declara **un solo canal de
color** y se reusa, como `X_OV` e `Y_LOG`: gris para 1 estrella, que son los arqueros, y coral
para 5, que son los que la lectura ingenua pone en el centro.

In [47]:
# un canal de color y un canal x para todos los paneles
jug["est"] = jug["estrellas"].astype(str) + "★"            # 3 → "3★"
ORDEN   = ["1★", "2★", "3★", "4★", "5★"]
# domain y range fijan a mano qué color le toca a cada valor, en el mismo orden
ESC_EST = alt.Scale(domain=ORDEN, range=["#C9C9C9", "#7FE6DF", "#00C6BB", "#009086", "#EF755F"])
# sort=ORDEN impone ese orden en vez del alfabético
COLOR   = alt.Color("est:O", sort=ORDEN, legend=None, scale=ESC_EST)
# labelAngle=0 deja los rótulos del eje horizontales
EJE_EST = alt.X("est:O", sort=ORDEN, title="Gambeta (estrellas)", axis=alt.Axis(labelAngle=0))
GRIS, TINTA = "#636363", "#212121"

### 7.5 El texto es una capa

Una anotación no se dibuja encima del gráfico: es **otra capa**, con sus propios datos —una tabla
chica con el texto y dónde va—, su propia marca, `mark_text`, y su propio mapeo, en las mismas
coordenadas que los datos. Los tres paneles se arman así.

In [48]:
# panel 1 · lo que parece: las barras, el número sobre cada una y el ×16
med = jug.groupby("est", as_index=False)["valor"].median()
barras   = alt.Chart(med).mark_bar().encode(x=EJE_EST, y=alt.Y("valor:Q", title="Valor mediano (M€)"), color=COLOR)
# barras.mark_text reusa el gráfico de las barras con otra marca: mismos datos y mismo mapeo
# dy=-8 corre el texto 8 píxeles hacia arriba; format=".2f" lo muestra con dos decimales
# color=alt.value(TINTA) pisa el color que venía de las barras
numeros  = barras.mark_text(dy=-8, color=TINTA).encode(text=alt.Text("valor:Q", format=".2f"), color=alt.value(TINTA))
cociente = alt.Chart(pd.DataFrame({"x": ["3★"], "y": [9.5], "t": ["×16 entre 2★ y 5★"]})).mark_text(
    color="#EF755F", fontSize=15, fontWeight="bold").encode(x=alt.X("x:O", sort=ORDEN), y="y:Q", text="t:N")

panel1 = (barras + numeros + cociente).properties(width=300, height=280)
panel1

alt.LayerChart(...)

In [49]:
# panel 2 · lo que lo desarma: el overall por estrellas, con tres anotaciones
# size=32 es el ancho de cada caja; median y outliers cambian cómo se dibujan la mediana y los casos extremos
cajas = alt.Chart(jug[["est", "overall"]]).mark_boxplot(size=32, median={"color": TINTA}, outliers={"size": 5, "opacity": 0.3}).encode(
    x=EJE_EST, y=alt.Y("overall:Q", title="Overall", scale=alt.Scale(domain=[40, 100])), color=COLOR)
notas = alt.Chart(pd.DataFrame({"x": ["1★", "2★", "5★"], "y": [43, 96, 96],
                                "t": ["todos arqueros", "mediana 64", "mediana 77"]})).mark_text(
    color=GRIS, fontSize=11).encode(x=alt.X("x:O", sort=ORDEN), y="y:Q", text="t:N")

panel2 = (cajas + notas).properties(width=300, height=280)
panel2

alt.LayerChart(...)

In [50]:
# panel 3 · la respuesta: las curvas, una regla en overall 80, su lectura y los rótulos de cada curva
curvas["est"] = curvas["estrellas"].astype(str) + "★"
X_P3  = alt.X("overall:Q", title="Overall", scale=alt.Scale(domain=[60, 84]))
X_TXT = alt.X("x:Q", title="Overall")

# point=alt.OverlayMarkDef(...) agrega los puntos sobre la línea con su propio tamaño; clip=True recorta lo que sale del eje
lineas  = alt.Chart(curvas).mark_line(strokeWidth=3, clip=True, point=alt.OverlayMarkDef(size=25, clip=True)).encode(
    x=X_P3, y=alt.Y("valor:Q", title="Valor mediano (M€, escala log)", scale=alt.Scale(type="log")), color=COLOR)
# una regla con sólo x es una línea vertical de punta a punta; strokeDash=[4, 3]: 4 píxeles de trazo y 3 de hueco
regla   = alt.Chart(pd.DataFrame({"x": [80]})).mark_rule(strokeDash=[4, 3], color=GRIS).encode(x=X_TXT)
# align="right": el texto termina en x, en vez de empezar ahí
lectura = alt.Chart(pd.DataFrame({"x": [79.4, 79.4], "y": [1.2, 0.85], "t": ["a overall 80: de 20 a 24 M€", "con 2, 3 o 4 estrellas"]})).mark_text(
    align="right", color=GRIS, fontSize=11).encode(x=X_TXT, y="y:Q", text="t:N")
rotulos = alt.Chart(pd.DataFrame({"x": [61] * 3, "y": [30, 19, 12], "est": ["2★", "3★", "4★"]})).mark_text(
    align="left", fontSize=13, fontWeight="bold").encode(x=X_TXT, y="y:Q", text="est:N", color=COLOR)

panel3 = (lineas + regla + lectura + rotulos).properties(width=300, height=280)
panel3

alt.LayerChart(...)

En el panel 3 los rótulos de cada curva reemplazan a la leyenda: son una capa de texto que usa
**el mismo canal de color** que las líneas, y por eso cada rótulo sale del color de su curva.

### 7.6 Títulos que afirman

El título también es una declaración, y puede ser de dos tipos. Uno que repite la pregunta
(«Valor según estrellas») obliga al lector a encontrar la respuesta solo. Uno que **afirma** («La
gambeta no se paga: se paga el nivel») se la da, y el gráfico la prueba. Cada panel lleva el suyo,
con un subtítulo que dice qué mirar.

In [51]:
# un título con subtítulo, como propiedad del panel
# subtitle acepta una lista: cada elemento es un renglón; anchor="start" alinea el título a la izquierda
def con_titulo(panel, titulo, subtitulo):
    return panel.properties(title=alt.Title(titulo, subtitle=subtitulo, anchor="start", fontSize=16, subtitleColor=GRIS))

con_titulo(panel2, "2 · La trampa", ["Los de más estrellas también son los mejores.", "Y 1★ no es un jugador torpe: es un arquero."])

alt.LayerChart(...)

### 7.7 La composición

In [52]:
# tres paneles concatenados, el título general y la configuración común
# spacing: píxeles entre paneles; offset: distancia entre el título y los paneles
# :, separa los miles con coma y .replace la cambia por punto
# configure_* vale para todo el póster: padding es el margen, stroke=None saca el borde de cada panel,
# y configure_axis da el mismo estilo a todos los ejes
poster = alt.hconcat(
    con_titulo(panel1, "1 · Lo que parece", ["Cuantas más estrellas de gambeta,", "más vale el jugador."]),
    con_titulo(panel2, "2 · La trampa", ["Los de más estrellas también son los mejores.", "Y 1★ no es un jugador torpe: es un arquero."]),
    con_titulo(panel3, "3 · La respuesta", ["Sin arqueros y a igual overall,", "las tres curvas se pisan."]),
    spacing=40
).properties(title=alt.Title(
    "La gambeta no se paga: se paga el nivel",
    subtitle=["Un jugador de 5★ vale 16 veces más que uno de 2★, pero también es mucho mejor. A igual overall, el mercado no paga extra por la gambeta.",
              f"FIFA · sofifa.com · {len(jug):,} jugadores".replace(",", ".") + " · mediana por grupo · en el panel 3, sin arqueros y sólo grupos de 20 jugadores o más (los 5★ no llegan)"],
    fontSize=26, subtitleFontSize=13, anchor="start", offset=18, subtitleColor=GRIS)
).configure(padding=24).configure_view(stroke=None).configure_axis(
    labelColor=GRIS, titleColor=GRIS, gridColor="#F0F0F0", titleFontWeight="normal")

poster

alt.HConcatChart(...)

In [53]:
# se guarda tal como se ve; scale_factor=2 duplica la resolución (usa vl-convert, instalado al principio)
poster.save("poster_gambeta.png", scale_factor=2)

Es un **póster particionado**: tres paneles que el lector recorre de izquierda a derecha, con la
historia contada por el autor. Y es una sola declaración. Cada decisión se puede señalar en el
código y tiene su sección: los datos sin valor cero (1.1), el tipo ordinal (2.3), la mediana
(3.1), el filtro (3.3), la escala log (4.1), las capas de texto (5.1 y 7.5), la concatenación
(5.4) y un canal de color compartido (7.4).

### 7.8 La misma historia, otra oración

Tres paneles no son la única forma. La misma historia entra en **un solo gráfico anotado**: la
curva de lo que vale un jugador de campo de 2 a 4 estrellas según su overall, y encima los de 5
estrellas, uno por uno. Si la gambeta se pagara, los puntos quedarían por encima de la curva.

In [54]:
# la curva de referencia: jugadores de campo de 2 a 4 estrellas, por overall (al menos 5 por punto)
# between(2, 4) incluye los dos extremos
ref = (jug[(jug["rol"] != "Arquero") & jug["estrellas"].between(2, 4)]
       .groupby("overall")["valor"].agg(valor="median", n="size").reset_index())
ref = ref[ref["n"] >= 5]

# los de 5 estrellas, cada uno con la referencia de su overall
# merge pega a cada jugador el valor de referencia de su mismo overall
cinco = (jug[(jug["estrellas"] == 5) & (jug["rol"] != "Arquero")][["nombre", "overall", "valor"]]
         .merge(ref[["overall", "valor"]].rename(columns={"valor": "referencia"}), on="overall"))
cinco["cociente"] = cinco["valor"] / cinco["referencia"]
# describe da varias estadísticas; se muestran la cantidad y la mediana (50%)
cinco["cociente"].describe()[["count", "50%"]].round(2)

,cociente
count,51.00
50%,1.08


In [55]:
# qué parte de los 5★ vale más que la referencia de su overall
(cinco["cociente"] > 1).mean().round(2)

np.float64(0.53)

En mediana, un jugador de 5 estrellas vale 1,08 veces lo que uno de 2 a 4 con su mismo overall,
y la mitad queda por encima y la mitad por debajo. Eso es lo que el gráfico tiene que mostrar.

In [56]:
# tres capas: la curva, los 5★ y el texto, que hace de leyenda y de conclusión
x_ov  = alt.X("overall:Q", title="Overall", scale=alt.Scale(domain=[60, 92]))
y_val = alt.Y("valor:Q", title="Valor de mercado (M€, escala log)", scale=alt.Scale(type="log"))

curva_ref = alt.Chart(ref).mark_line(strokeWidth=3, color="#009086", clip=True).encode(x=x_ov, y=y_val)
cracks    = alt.Chart(cinco).mark_circle(size=70, color="#EF755F", opacity=0.85).encode(
    x=x_ov, y=y_val, tooltip=["nombre", "overall", "valor"])
# scale=None en el color: los valores de la columna c se usan tal cual, como colores
texto     = alt.Chart(pd.DataFrame({
    "x": [61, 61, 74, 74], "y": [300, 150, 0.35, 0.22],
    "t": ["● cada punto es uno de los 51 jugadores de 5★",
          "— lo que vale uno de 2★ a 4★ con el mismo overall",
          "a igual overall, los de 5★ valen 1,08 veces más:",
          "la mitad queda arriba de la curva y la mitad abajo"],
    "c": ["#EF755F", "#009086", TINTA, TINTA]})).mark_text(align="left", fontSize=12).encode(
    x=alt.X("x:Q", title="Overall"), y="y:Q", text="t:N", color=alt.Color("c:N", scale=None))

(curva_ref + cracks + texto).properties(width=620, height=360, title=alt.Title(
    "La gambeta no se paga: se paga el nivel",
    subtitle="Los de 5★ valen 16 veces más que los de 2★ porque son mejores; a igual overall, valen casi lo mismo.",
    anchor="start", fontSize=20, subtitleColor=GRIS)).configure_view(stroke=None)

alt.LayerChart(...)

Es otro género, el **gráfico anotado**, y otra oración: capas en lugar de concatenación. Gana
densidad —la trampa y la respuesta están en el mismo lugar— y pierde el recorrido: el lector no
ve primero la lectura ingenua, así que el subtítulo tiene que contarla. Ninguna de las dos formas
es la correcta; elegir una es otra decisión del autor.

### 7.9 La misma historia, para explorar: una copa de martini

El póster y el gráfico anotado son **guiados por el autor**: el lector ve lo que se eligió, en el
orden en que se eligió. La **copa de martini** cuenta la misma historia en pasos —el tallo— y al
final se abre para que el lector explore —la copa—. En Altair no hace falta otra herramienta: el
paso es un **parámetro** atado a unos botones, y cada capa del gráfico declara en qué paso se ve.

In [57]:
# los puntos: la muestra de siempre, más todos los de 5★ (en la muestra hay pocos)
# drop_duplicates evita que un 5★ que ya estaba en la muestra aparezca dos veces
puntos = pd.concat([muestra, jug[jug["estrellas"] == 5]]).drop_duplicates(subset=["nombre", "overall", "valor", "edad"])
puntos = puntos.assign(est=puntos["estrellas"].astype(str) + "★")[["nombre", "rol", "est", "estrellas", "overall", "valor"]]

# lo que muestra cada paso, calculado con pandas
med_valor = jug.groupby("est", as_index=False)["valor"].median()                                           # paso 1
med_overall = jug.groupby("est", as_index=False)["overall"].median()                                       # paso 2
pd.concat([med_valor.set_index("est"), med_overall.set_index("est"), puntos["est"].value_counts().rename("en el gráfico")], axis=1)

,valor,overall,en el gráfico
est,,,
1★,0.500,64.0,208
2★,0.775,64.0,969
3★,1.400,67.0,691
4★,3.100,72.0,127
5★,12.500,77.0,53


Tres parámetros: el paso, con botones, y dos menús para el último paso. Ninguno hace nada por sí
solo; lo que importa es quién los lee.

In [58]:
# los parámetros: el paso (botones) y dos filtros para explorar
# name= les pone un nombre fijo, para poder leerlos por ese nombre dentro de las expresiones de texto
# binding_radio arma botones: options son los valores y labels lo que se lee en cada botón
PASOS = ["1 · Lo que parece", "2 · La trampa", "3 · La respuesta", "4 · Explorá"]
paso = alt.param(name="paso", value=1, bind=alt.binding_radio(options=[1, 2, 3, 4], labels=PASOS, name="Paso: "))
rol_sel = alt.param(name="rol_sel", value="Todos",
                    bind=alt.binding_select(options=["Todos"] + sorted(jug["rol"].unique()), name="Rol (paso 4): "))
est_sel = alt.param(name="est_sel", value="Todas",
                    bind=alt.binding_select(options=["Todas"] + ORDEN, name="Estrellas (paso 4): "))

Las capas son las de siempre —puntos, reglas, líneas, texto—, cada una con sus datos. Lo nuevo es
que la **opacidad** de cada una sale de una expresión que lee el paso: las reglas de las medianas
sólo existen en el paso 1, las del overall en el 2, las curvas desde el 3. Los puntos están
siempre, pero su opacidad depende del paso y, en el 4, de los dos menús.

In [59]:
# ejes fijos para todos los pasos: con el dominio fijo, el gráfico no salta al cambiar de paso
X_M = alt.X("overall:Q", title="Overall", scale=alt.Scale(domain=[45, 93]))
Y_M = alt.Y("valor:Q", title="Valor de mercado (M€, escala log)", scale=alt.Scale(type="log", domain=[0.02, 400]))

# los jugadores: su opacidad depende del paso, y en el paso 4 de los menús
# las expresiones leen los parámetros por su nombre (paso, rol_sel, est_sel); || es «o» y && es «y»
# scale=None en opacity y size: el número calculado se usa tal cual, y legend=None no arma leyenda
jugadores = alt.Chart(puntos).transform_calculate(
    op="paso <= 2 ? 0.3 : paso == 3 ? (datum.estrellas == 5 ? 0.95 : 0.07) : "
       "((rol_sel == 'Todos' || datum.rol == rol_sel) && (est_sel == 'Todas' || datum.est == est_sel)) ? 0.8 : 0.03",
    tam="paso == 3 && datum.estrellas == 5 ? 70 : 18"
).mark_circle().encode(
    x=X_M, y=Y_M, color=alt.Color("est:O", sort=ORDEN, scale=ESC_EST, title="Estrellas"),
    opacity=alt.Opacity("op:Q", scale=None, legend=None), size=alt.Size("tam:Q", scale=None, legend=None),
    tooltip=["nombre", "rol", "est", "overall", alt.Tooltip("valor:Q", format=".2f")])

# paso 1 · la mediana del valor de cada grupo, como una línea horizontal con su rótulo
reglas_valor = alt.Chart(med_valor).transform_calculate(op="paso == 1 ? 1 : 0").mark_rule(strokeWidth=2).encode(
    y=Y_M, color=alt.Color("est:O", sort=ORDEN, scale=ESC_EST, title="Estrellas"), opacity=alt.Opacity("op:Q", scale=None))
# format y replace son funciones de las expresiones: dos decimales, y coma en lugar de punto
rotulos_valor = alt.Chart(med_valor).transform_calculate(op="paso == 1 ? 1 : 0", x="92.5",
    t="datum.est + ': ' + replace(format(datum.valor, '.2f'), '.', ',') + ' M€'").mark_text(align="right", dy=-6, fontSize=11, fontWeight="bold").encode(
    x=alt.X("x:Q", scale=alt.Scale(domain=[45, 93])), y=Y_M, text="t:N",
    color=alt.Color("est:O", sort=ORDEN, scale=ESC_EST, title="Estrellas"), opacity=alt.Opacity("op:Q", scale=None))

# paso 2 · el overall mediano de cada grupo, como una línea vertical
# y e y2 marcan dónde empieza y dónde termina la regla; dx corre el texto en horizontal
reglas_overall = alt.Chart(med_overall[med_overall["est"] != "1★"].assign(y0=0.02, y1=60)).transform_calculate(op="paso == 2 ? 1 : 0").mark_rule(strokeWidth=2).encode(
    x=X_M, y=alt.Y("y0:Q", scale=alt.Scale(type="log", domain=[0.02, 400])), y2="y1:Q", color=alt.Color("est:O", sort=ORDEN, scale=ESC_EST, title="Estrellas"), opacity=alt.Opacity("op:Q", scale=None))
rotulos_overall = alt.Chart(med_overall[med_overall["est"] != "1★"].assign(y=[0.045, 0.03, 0.045, 0.03])).transform_calculate(
    op="paso == 2 ? 1 : 0", t="datum.est + ': ' + datum.overall").mark_text(align="left", dx=4, fontSize=11, fontWeight="bold").encode(
    x=X_M, y=alt.Y("y:Q", scale=alt.Scale(type="log", domain=[0.02, 400])), text="t:N",
    color=alt.Color("est:O", sort=ORDEN, scale=ESC_EST, title="Estrellas"), opacity=alt.Opacity("op:Q", scale=None))

# paso 3 en adelante · las curvas de la sección 7.3
lineas_m = alt.Chart(curvas).transform_calculate(op="paso >= 3 ? 0.9 : 0").mark_line(strokeWidth=3).encode(
    x=X_M, y=Y_M, color=alt.Color("est:O", sort=ORDEN, scale=ESC_EST, title="Estrellas"), opacity=alt.Opacity("op:Q", scale=None))

# el relato: dos líneas de texto por paso, filtradas por el parámetro
# el filtro también puede escribirse como texto: se queda con las filas del paso elegido
RELATO = pd.DataFrame({"p": [1, 1, 2, 2, 3, 3, 4, 4], "y": [250, 140] * 4, "t": [
    "Lo que parece: de 2★ a 5★, el valor mediano",
    "pasa de 0,78 a 12,5 M€. Dieciséis veces.",
    "La trampa: los de más estrellas también son mejores",
    "(overall 64 → 77). Y los de 1★ son todos arqueros.",
    "La respuesta: a igual overall, 2★, 3★ y 4★ valen lo mismo,",
    "y los de 5★ caen sobre las mismas curvas.",
    "Explorá: pasá el mouse por un jugador, filtrá por rol",
    "o por estrellas, y acercate con la rueda."]})
relato = alt.Chart(RELATO).transform_filter("datum.p == paso").transform_calculate(x="46").mark_text(
    align="left", fontSize=14, color=TINTA).encode(
    x=alt.X("x:Q", scale=alt.Scale(domain=[45, 93])), y=alt.Y("y:Q", scale=alt.Scale(type="log", domain=[0.02, 400])), text="t:N")

Se juntan las capas, se agregan los parámetros y se le da al conjunto el zoom de la sección 6.1.
Los botones aparecen debajo del gráfico: el paso 1 es la lectura ingenua, y del 4 en adelante el
lector manda.

In [60]:
# la copa de martini: siete capas y tres parámetros
# alt.layer(a, b, c) es lo mismo que a + b + c; los parámetros se agregan una sola vez, al conjunto
copa = alt.layer(jugadores, reglas_valor, rotulos_valor, reglas_overall, rotulos_overall, lineas_m, relato).add_params(
    paso, rol_sel, est_sel
).properties(width=640, height=400, title=alt.Title(
    "La gambeta no se paga: se paga el nivel", subtitle="Recorré los pasos con los botones de abajo.",
    anchor="start", fontSize=20, subtitleColor=GRIS)).configure_view(stroke=None).interactive()

copa

alt.LayerChart(...)

In [61]:
# se guarda como una página web, para abrirla en el navegador o compartirla
# la página baja de internet las bibliotecas que dibujan el gráfico; con inline=True
# quedan adentro del archivo y se abre sin conexión (también usa vl-convert)
copa.save("copa_de_martini.html")

Las cuatro vistas son una sola declaración: el paso no cambia el gráfico, cambia valores que las
capas leen. Es la estructura que pide la Entrega 4 del Integrador sobre los datos propios: un
tallo con la historia que encontraste, y una copa donde el lector puede hacer sus preguntas.

## Cierre

Un gráfico de Altair es una declaración: una tabla, una marca, un mapeo de variables a atributos
visuales, y entre los datos y la marca, transformaciones y escalas. Componer —capas, facetas,
concatenaciones— permite pasar de un gráfico a un póster, y los parámetros permiten pasar de un
póster a una herramienta. Ninguna de esas piezas es automática: cada una es una decisión, y el
póster de la Parte 7 sólo dijo algo cierto porque se tomaron mirando los datos.

Esa es la frontera de la Entrega 4 del Integrador: empezar con la historia que encontraste en tus
datos y después dejar que el lector haga sus propias preguntas, como en la copa de martini de la
sección 7.9.